In [ ]:
%pip install scikit-learn pandas sentence-transformers

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Intentamos cargar Sentence-Transformers para similitud semántica real (embeddings).
# Si no está instalado o hay límite de memoria en el despliegue, usa TF-IDF automáticamente.
try:
    from sentence_transformers import SentenceTransformer
    EMBEDDINGS_DISPONIBLES = True
except ImportError:
    EMBEDDINGS_DISPONIBLES = False

# Base de conocimiento de Nexia Analytics & Software

In [ ]:
def cargar_base_conocimiento() -> pd.DataFrame:
    datos = [
        # --- Políticas ---
        {"id": "DOC-01", "categoria": "Política", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "GitHub",
         "texto": "Guía de estilo y buenas prácticas para escribir código limpio (Clean Code) en Python."},
        {"id": "DOC-02", "categoria": "Política", "actividad": "Implementación_Machine_Learning", "tecnologia": "Base_Conocimiento",
         "texto": "Lineamientos de gobernanza de datos y privacidad de la información del cliente."},
        {"id": "DOC-03", "categoria": "Política", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "GitHub",
         "texto": "Reglamento interno y políticas de la empresa para el uso de GitHub y repositorios."},
        {"id": "DOC-04", "categoria": "Política", "actividad": "Resolución_Incidentes_Críticos", "tecnologia": "Entorno_Cloud",
         "texto": "Políticas de seguridad de la información, manejo de contraseñas y accesos VPN."},
        {"id": "DOC-05", "categoria": "Política", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "Base_Conocimiento",
         "texto": "Reglamento de confidencialidad y acuerdos de no divulgación (NDA) para consultores."},
        {"id": "DOC-06", "categoria": "Política", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "GitHub",
         "texto": "Lineamientos estándar para la revisión de código (Pull Requests) entre pares."},
        {"id": "DOC-07", "categoria": "Política", "actividad": "Resolución_Incidentes_Críticos", "tecnologia": "Base_Conocimiento",
         "texto": "Normativa oficial de la empresa sobre horarios laborales y uso de equipos físicos."},
        {"id": "DOC-08", "categoria": "Política", "actividad": "Resolución_Incidentes_Críticos", "tecnologia": "Base_Conocimiento",
         "texto": "Políticas de uso aceptable de dispositivos personales y corporativos."},

        # --- Técnicos ---
        {"id": "DOC-09", "categoria": "Técnico", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "Entorno_Cloud",
         "texto": "Manual de arquitectura de software para el nuevo cliente corporativo."},
        {"id": "DOC-10", "categoria": "Técnico", "actividad": "Implementación_Machine_Learning", "tecnologia": "Base_Conocimiento",
         "texto": "Diccionario de datos para el pipeline del modelo de Machine Learning."},
        {"id": "DOC-11", "categoria": "Técnico", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "GitHub",
         "texto": "Documentación técnica de la API REST para el manejo de usuarios."},
        {"id": "DOC-12", "categoria": "Técnico", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "Entorno_Cloud",
         "texto": "Diagramas de flujo de datos y arquitectura de microservicios en AWS."},
        {"id": "DOC-13", "categoria": "Técnico", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "Entorno_Cloud",
         "texto": "Esquema relacional de la base de datos PostgreSQL para el módulo de facturación."},
        {"id": "DOC-14", "categoria": "Técnico", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "Entorno_Cloud",
         "texto": "Especificaciones de los endpoints de integración con la pasarela de pagos."},

        # --- Operativos ---
        {"id": "DOC-15", "categoria": "Operativo", "actividad": "Resolución_Incidentes_Críticos", "tecnologia": "Entorno_Cloud",
         "texto": "Reporte post-mortem del fallo en el clúster de producción de AWS."},
        {"id": "DOC-16", "categoria": "Operativo", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "Contenedor",
         "texto": "Guía de despliegue de contenedores utilizando Docker y Kubernetes."},
        {"id": "DOC-17", "categoria": "Operativo", "actividad": "Resolución_Incidentes_Críticos", "tecnologia": "Entorno_Cloud",
         "texto": "Ticket de resolución de incidente crítico de latencia en la base de datos."},
        {"id": "DOC-18", "categoria": "Operativo", "actividad": "Diseño_Arquitectura_Software", "tecnologia": "Base_Conocimiento",
         "texto": "Manual de usuario paso a paso para el sistema interno de gestión de viáticos."},
        {"id": "DOC-19", "categoria": "Operativo", "actividad": "Resolución_Incidentes_Críticos", "tecnologia": "Entorno_Cloud",
         "texto": "Bitácora de mantenimiento semanal y rotación de credenciales en servidores de desarrollo."},
        {"id": "DOC-20", "categoria": "Operativo", "actividad": "Resolución_Incidentes_Críticos", "tecnologia": "Base_Conocimiento",
         "texto": "Reporte mensual de métricas de incidencias (SLA) y tiempos de respuesta de soporte."},
        {"id": "DOC-21", "categoria": "Operativo", "actividad": "Resolución_Incidentes_Críticos", "tecnologia": "Contenedor",
         "texto": "Guía rápida de soporte para reiniciar servicios y recuperar aplicaciones web caídas."},
    ]
    return pd.DataFrame(datos)

# Motor de recomendación

In [ ]:
class SistemaRecomendacionNexia:
    def __init__(self, df_docs: pd.DataFrame, usar_embeddings: bool = True):
        self.df = df_docs.reset_index(drop=True).copy()
        self.usar_embeddings = usar_embeddings and EMBEDDINGS_DISPONIBLES

        # Combinamos texto + categoría + tecnología para enriquecer la representación semántica
        corpus_enriquecido = (
            self.df["categoria"] + ": " + self.df["texto"] + " (" + self.df["tecnologia"] + ")"
        ).tolist()

        if self.usar_embeddings:
            # Modelo ligero multilingüe (soporta español e inglés técnico)
            self.modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
            self.vectores_docs = self.modelo.encode(corpus_enriquecido, normalize_embeddings=True)
            self.modo = "Embeddings Semánticos (Sentence-BERT)"
        else:
            stop_words_es = [
                "de", "la", "el", "los", "las", "y", "en", "para", "del", "al",
                "un", "una", "con", "por", "sobre", "entre", "o", "a", "que", "se", "su", "sus", "es"
            ]
            self.vectorizador = TfidfVectorizer(
                stop_words=stop_words_es,
                strip_accents="unicode",
                ngram_range=(1, 2)
            )
            self.vectores_docs = self.vectorizador.fit_transform(corpus_enriquecido).toarray()
            self.modo = "TF-IDF (N-gramas)"

        # Matriz precalculada de similitud documento-a-documento (N x N)
        self.matriz_similitud_docs = cosine_similarity(self.vectores_docs)

    def _vectorizar_textos(self, textos: list[str]) -> np.ndarray:
        if self.usar_embeddings:
            return self.modelo.encode(textos, normalize_embeddings=True)
        return self.vectorizador.transform(textos).toarray()

    def agregar_documento(self, texto: str, categoria: str, actividad: str = "General", tecnologia: str = "Base_Conocimiento"):
        """Permite conectar el clasificador automático para indexar documentos nuevos en caliente."""
        nuevo_id = f"DOC-{len(self.df) + 1:02d}"
        nueva_fila = pd.DataFrame([{
            "id": nuevo_id, "categoria": categoria, "actividad": actividad,
            "tecnologia": tecnologia, "texto": texto
        }])
        self.df = pd.concat([self.df, nueva_fila], ignore_index=True)
        self.__init__(self.df, usar_embeddings=self.usar_embeddings)

    # MECANISMO A: Recomendación por similitud semántica entre documentos
    def recomendar_por_documento(self, doc_id: str, top_k: int = 3, boost_ontologia: float = 0.05) -> pd.DataFrame:
        """
        Dado un documento seleccionado/abierto por el usuario, sugiere los top_k
        documentos más parecidos semánticamente, aplicando un ligero impulso
        si comparten la misma Actividad o Tecnología en la ontología.
        """
        if doc_id not in self.df["id"].values:
            raise ValueError(f"El ID '{doc_id}' no existe en la base de conocimiento.")

        idx = self.df.index[self.df["id"] == doc_id][0]
        doc_base = self.df.iloc[idx]

        scores = self.matriz_similitud_docs[idx].copy()

        # Refuerzo ontológico ligero (misma actividad o misma herramienta)
        misma_actividad = (self.df["actividad"] == doc_base["actividad"]).astype(float) * boost_ontologia
        misma_tecnologia = (self.df["tecnologia"] == doc_base["tecnologia"]).astype(float) * boost_ontologia
        scores_finales = scores + misma_actividad + misma_tecnologia

        resultados = self.df.copy()
        resultados["similitud_semantica"] = scores
        resultados["score_final"] = scores_finales

        # Excluir el mismo documento consultado
        resultados = resultados[resultados["id"] != doc_id]
        return resultados.sort_values("score_final", ascending=False).head(top_k)

    
    # MECANISMO B: Recomendación basada en historial de consultas y contexto
    def recomendar_por_contexto(
        self,
        historial_consultas: list[str],
        docs_vistos: list[str] | None = None,
        top_k: int = 3,
        umbral: float = 0.15
    ) -> pd.DataFrame:
        """
        Construye un vector de interés del usuario a partir de:
        1) Su historial de búsquedas (con decaimiento exponencial: las consultas
           más recientes tienen mayor peso).
        2) Los documentos ya abiertos en la sesión (filtrándolos para no repetir).
        """
        docs_vistos = docs_vistos or []
        vectores_contexto = []
        pesos = []

        # 1. Ponderar consultas recientes (más peso a las últimas consultas)
        if historial_consultas:
            vecs_q = self._vectorizar_textos(historial_consultas)
            n_q = len(historial_consultas)
            for i, vec in enumerate(vecs_q):
                peso_recencia = 0.6 ** (n_q - 1 - i)  # La más reciente vale 1.0, la anterior 0.6, etc.
                vectores_contexto.append(vec)
                pesos.append(peso_recencia)

        # 2. Incorporar documentos consultados previamente como señal implícita de interés
        for doc_id in docs_vistos:
            if doc_id in self.df["id"].values:
                idx = self.df.index[self.df["id"] == doc_id][0]
                vectores_contexto.append(self.vectores_docs[idx])
                pesos.append(0.5)

        if not vectores_contexto:
            return pd.DataFrame()

        # Vector perfil del usuario (promedio ponderado)
        perfil_usuario = np.average(vectores_contexto, axis=0, weights=pesos).reshape(1, -1)
        similitudes = cosine_similarity(perfil_usuario, self.vectores_docs).flatten()

        resultados = self.df.copy()
        resultados["afinidad_contexto"] = similitudes

        # Filtrar documentos ya vistos y los que no superan el umbral mínimo
        resultados = resultados[~resultados["id"].isin(docs_vistos)]
        resultados = resultados[resultados["afinidad_contexto"] >= umbral]

        return resultados.sort_values("afinidad_contexto", ascending=False).head(top_k)

# Pruebas y Validación

In [ ]:
# Inicialización del motor
df_nexia = cargar_base_conocimiento()
recomendador = SistemaRecomendacionNexia(df_nexia, usar_embeddings=True)
print(f"Motor activo: {recomendador.modo}\n")

# --- PRUEBA 1: Similitud semántica entre documentos ---
doc_actual = "DOC-15"  # Reporte post-mortem del fallo en el clúster de producción de AWS
info_doc = df_nexia[df_nexia["id"] == doc_actual].iloc[0]
print(f" Documento abierto: [{info_doc['id']}] ({info_doc['categoria']}) {info_doc['texto']}")
print(" Documentos recomendados relacionados:")
recs_doc = recomendador.recomendar_por_documento(doc_actual, top_k=3)
for _, row in recs_doc.iterrows():
    print(f"   • [{row['id']}] [{row['categoria']}] (Score: {row['score_final']:.2f}) -> {row['texto']}")

print("\n" + "-" * 75 + "\n")

# --- PRUEBA 2: Recomendación basada en historial de consultas y sesión ---
historial_usuario = [
    "cómo estructurar tablas de clientes",
    "modelos predictivos y variables de entrada"
]
documentos_ya_leidos = ["DOC-10"]  # Ya leyó el Diccionario de datos de ML

print(f" Historial de búsqueda: {historial_usuario}")
print(f" Documentos ya consultados en sesión: {documentos_ya_leidos}")
print(" Sugerencias personalizadas según tu contexto:")
recs_ctx = recomendador.recomendar_por_contexto(
    historial_consultas=historial_usuario,
    docs_vistos=documentos_ya_leidos,
    top_k=3
)
for _, row in recs_ctx.iterrows():
    print(f"   • [{row['id']}] [{row['categoria']}] (Afinidad: {row['afinidad_contexto']:.2f}) -> {row['texto']}")

In [ ]:
import re

class ChatbotNexia:
    def __init__(self, motor_recomendacion: SistemaRecomendacionNexia):
        self.motor = motor_recomendacion
        self.historial_consultas: list[str] = []

    def reiniciar_conversacion(self) -> str:
        self.historial_consultas.clear()
        return "Listo, limpié el historial de la conversación. ¿Sobre qué otro tema de Nexia quieres consultar?"

    def responder(self, mensaje_usuario: str) -> str:
        texto = mensaje_usuario.strip()
        if not texto:
            return "Escribe una pregunta o el ID de un documento (por ejemplo, `DOC-15`) para ayudarte."

        texto_lower = texto.lower()

        # --- REGLA 1: Comandos de reinicio de contexto ---
        if any(cmd in texto_lower for cmd in ["limpiar", "reiniciar", "nuevo tema", "borrar historial"]):
            return self.reiniciar_conversacion()

        # --- REGLA 2: Saludos ---
        if re.fullmatch(r"(hola|buenos dias|buenos días|buenas tardes|buenas noches|que tal|qué tal|hey)[!?. ]*", texto_lower):
            return (
                "¡Hola! Soy el asistente de conocimiento de **Nexia Analytics & Software**. "
                "Puedo ayudarte a encontrar políticas internas, documentación técnica o guías operativas.\n\n"
                "Ejemplos de lo que puedes preguntarme:\n"
                "- *¿Cómo desplegar contenedores en Docker y Kubernetes?*\n"
                "- *Políticas de seguridad y contraseñas*\n"
                "- *Muéstrame el DOC-15*"
            )

        # --- REGLA 3: Ayuda ---
        if texto_lower in ["ayuda", "help", "comandos", "que puedes hacer", "¿qué puedes hacer?"]:
            return (
                "Puedo buscar información por lenguaje natural usando contexto de la charla o buscar un documento exacto por su código (`DOC-01` al `DOC-21`). "
                "Si cambias de tema y quieres evitar que mezcle preguntas anteriores, escribe **reiniciar**."
            )

        # --- REGLA 4: Despedidas ---
        if any(desp in texto_lower for desp in ["gracias", "muchas gracias", "adios", "adiós", "bye", "hasta luego"]):
            return "¡Con gusto! Si necesitas consultar otro documento o procedimiento, aquí sigo."

        # --- REGLA 5: Consulta directa por ID de documento (Mecanismo A) ---
        match_doc = re.search(r"\bdoc-\d{2}\b", texto_lower)
        if match_doc:
            doc_id = match_doc.group(0).upper()
            try:
                doc_info = self.motor.df[self.motor.df["id"] == doc_id].iloc[0]
                relacionados = self.motor.recomendar_por_documento(doc_id, top_k=2)

                respuesta = (
                    f"Aquí tienes la información del **{doc_id}**:\n"
                    f"- **Categoría:** {doc_info['categoria']} | **Tecnología:** {doc_info['tecnologia']}\n"
                    f"- **Descripción:** {doc_info['texto']}\n\n"
                    f"**Documentos relacionados que te pueden servir:**\n"
                )
                for _, row in relacionados.iterrows():
                    respuesta += f"- **[{row['id']}]** ({row['categoria']}) — {row['texto']} *(Score: {row['score_final']:.2f})*\n"
                return respuesta

            except (IndexError, ValueError):
                return f"No encontré ningún documento con el identificador **{doc_id}**. Los IDs disponibles van del `DOC-01` al `DOC-21`."

        # --- RECUPERACIÓN DE CONTEXTO (Mecanismo B) ---
        self.historial_consultas.append(texto)
        # Usamos una ventana de las últimas 3 consultas para mantener contexto sin arrastrar temas viejos
        contexto_reciente = self.historial_consultas[-3:]

        resultados = self.motor.recomendar_por_contexto(
            historial_consultas=contexto_reciente,
            docs_vistos=[],  # Se deja vacío para no ocultar documentos si el usuario repite una consulta
            top_k=3,
            umbral=0.15
        )

        if resultados.empty:
            # Sacamos la consulta sin coincidencias para no ensuciar el vector futuro
            self.historial_consultas.pop()
            return (
                "No encontré documentos que superen el umbral de relevancia para esa consulta. "
                "Intenta incluir términos como *AWS, Docker, GitHub, API REST, PostgreSQL, seguridad, incidente* o *viáticos*."
            )

        respuesta = "Encontré estos documentos relevantes en la base de conocimiento:\n\n"
        for _, row in resultados.iterrows():
            respuesta += (
                f"- **[{row['id']}]** `{row['categoria']}` · `{row['tecnologia']}` "
                f"(Afinidad: **{row['afinidad_contexto']:.2f}**)\n"
                f"  {row['texto']}\n"
            )
        respuesta += "\n¿Quieres profundizar en alguno o consultar un ID específico?"
        return respuesta

In [ ]:
bot = ChatbotNexia(recomendador)

preguntas_sugeridas = {
    "1": "Hola",
    "2": "Se cayó un servicio, ¿qué reportes o guías tenemos?",
    "3": "¿Y cómo reinicio los servicios web caídos?",
    "4": "Muéstrame los Documentos de políticas internas",
    "5": "Políticas de seguridad",
    "6": "reiniciar"
}

print("=== ASISTENTE NEXIA (Opciones Rápidas + Consulta Libre) ===")
print("Puedes escribir el número de una duda o redactar tu propia pregunta:")
for clave, pregunta in preguntas_sugeridas.items():
    print(f"  [{clave}] {pregunta}")
print("  [0] Salir del chat\n" + "-" * 65)

while True:
    entrada = input("Elige un número (1-6), escribe tu duda o '0' para salir: ").strip()

    if entrada in ["0", "salir", "exit", "quit"]:
        print("Bot: ¡Hasta luego! Conversación finalizada.")
        break

    if not entrada:
        continue

    # Si ingresas un número válido del menú, usa esa pregunta; de lo contrario, toma tu texto libre
    if entrada in preguntas_sugeridas:
        mensaje = preguntas_sugeridas[entrada]
        print(f"Tú (Opción {entrada}): {mensaje}")
    else:
        mensaje = entrada
        print(f"Tú: {mensaje}")

    respuesta = bot.responder(mensaje)
    print(f"Bot:\n{respuesta}\n" + "-" * 65)

# Knowledge graph visualization

In [ ]:
%pip install networkx pyvis

In [ ]:
import networkx as nx
from pyvis.network import Network
from IPython.display import IFrame

COLOR_CAT = {"Política": "#e74c3c", "Técnico": "#3498db", "Operativo": "#2ecc71"}
COLOR_ACT = "#f39c12"   # actividades
COLOR_TEC = "#9b59b6"   # tecnologías

def limpiar(nombre: str) -> str:
    return nombre.replace("_", " ")

def construir_grafo(recomendador, vecinos_por_doc: int = 2) -> nx.Graph:
    df = recomendador.df
    G = nx.Graph()

    # --- Nodos de la ontología ---
    for cat, col in COLOR_CAT.items():
        G.add_node(f"cat:{cat}", label=cat, title=f"Categoría: {cat}",
                   color=col, size=45, shape="dot")
    for act in df["actividad"].unique():
        G.add_node(f"act:{act}", label=limpiar(act), title=f"Actividad: {limpiar(act)}",
                   color=COLOR_ACT, size=32, shape="triangle")
    for tec in df["tecnologia"].unique():
        G.add_node(f"tec:{tec}", label=limpiar(tec), title=f"Tecnología: {limpiar(tec)}",
                   color=COLOR_TEC, size=32, shape="square")

    # --- Documentos y sus relaciones con la ontología ---
    for _, fila in df.iterrows():
        G.add_node(
            fila["id"], label=fila["id"], shape="dot", size=16,
            color=COLOR_CAT[fila["categoria"]],
            title=(f"{fila['id']} · {fila['categoria']}\n{fila['texto']}\n"
                   f"Actividad: {limpiar(fila['actividad'])}\n"
                   f"Tecnología: {limpiar(fila['tecnologia'])}"),
        )
        G.add_edge(fila["id"], f"cat:{fila['categoria']}", title="pertenece_a", color="#777777")
        G.add_edge(fila["id"], f"act:{fila['actividad']}", title="parte_de_actividad", color=COLOR_ACT)
        G.add_edge(fila["id"], f"tec:{fila['tecnologia']}", title="usa_tecnología", color=COLOR_TEC)

    # --- Similitud semántica entre documentos (usa la matriz del recomendador) ---
    sim = recomendador.matriz_similitud_docs
    for i in range(len(df)):
        orden = sim[i].argsort()[::-1]
        for j in [k for k in orden if k != i][:vecinos_por_doc]:
            a, b = df.iloc[i]["id"], df.iloc[j]["id"]
            if not G.has_edge(a, b):
                G.add_edge(a, b, title=f"similar_a ({float(sim[i, j]):.2f})",
                           color="#bbbbbb", dashes=True, width=1 + 2 * float(sim[i, j]))
    return G

G = construir_grafo(recomendador, vecinos_por_doc=2)
print(f"Nodos: {G.number_of_nodes()} | Aristas: {G.number_of_edges()}")

In [ ]:
def dibujar(G, archivo="grafo_conocimiento.html", doc_foco=None, recomendador=None):
    H = G.copy()

    # Resalta un documento y lo que recomienda el motor (Mecanismo A)
    if doc_foco:
        recs = recomendador.recomendar_por_documento(doc_foco, top_k=3)["id"].tolist()
        for n, borde in [(doc_foco, "#ffffff")] + [(r, "#ffd700") for r in recs]:
            H.nodes[n]["color"] = {"background": H.nodes[n]["color"], "border": borde}
            H.nodes[n]["borderWidth"] = 4
            H.nodes[n]["size"] = 26
        for r in recs:
            H.add_edge(doc_foco, r, color="#ff4d4d", width=4, title="recomendado")

    net = Network(height="780px", width="100%", bgcolor="#1e1e1e",
                  font_color="white", cdn_resources="in_line")
    net.from_nx(H)
    net.force_atlas_2based(gravity=-60, spring_length=110)

    # Guardar en UTF-8 (evita el UnicodeEncodeError en Windows)
    with open(archivo, "w", encoding="utf-8") as f:
        f.write(net.generate_html())

    return IFrame(archivo, width="100%", height=800)

# Grafo completo
dibujar(G)

In [ ]:
dibujar(G, archivo="grafo_doc15.html", doc_foco="DOC-15", recomendador=recomendador)

In [ ]:
tipos = {"cat:": "Categorías", "act:": "Actividades", "tec:": "Tecnologías"}
for prefijo, nombre in tipos.items():
    print(f"\n{nombre} (documentos conectados):")
    nodos = [n for n in G if n.startswith(prefijo)]
    for n in sorted(nodos, key=lambda x: sum(1 for v in G.neighbors(x) if v.startswith("DOC")), reverse=True):
        print(f"  {G.nodes[n]['label']}: {sum(1 for v in G.neighbors(n) if v.startswith('DOC'))}")

centralidad = nx.betweenness_centrality(G)
puentes = [n for n in sorted(centralidad, key=centralidad.get, reverse=True) if n.startswith("DOC")][:3]
print("\nDocumentos más centrales (puentes entre categorías):", puentes)